# Baseline 2: Zero-Shot LLM via Groq API (Llama-3)

In this notebook, we evaluate a modern Large Language Model (LLM) without any task-specific fine-tuning (Zero-Shot). We use the `Llama-3-8b` model accessed via the Groq API.

**Objective:** Test whether a generalized LLM can natively distinguish between true toxicity and typical multiplayer game banter based solely on its pre-trained knowledge and a carefully engineered prompt.

In [ ]:
import os
import pandas as pd
from groq import Groq
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm.auto import tqdm
import time

os.environ["GROQ_API_KEY"] = "xxxxxxxxxxxxx" # Cencored for security reasons. Please set your own API key in the environment variable GROQ_API_KEY.


# Initialize the Groq client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

print("Loading and splitting data...")
df = pd.read_csv('../data/train.csv')[['comment_text', 'toxic']].dropna()

# Split the data identically to previous baselines to ensure a fair comparison
X_train, X_test, y_train, y_test = train_test_split(
    df['comment_text'], df['toxic'], test_size=0.2, random_state=42
)

# Sample 100 instances to avoid hitting API rate limits and to speed up local testing
print("Sampling 100 comments for Zero-Shot API testing...")
X_test_sampled = X_test.sample(n=100, random_state=42)
y_test_sampled = y_test.loc[X_test_sampled.index]

Loading and splitting data...
Sampling 100 comments for Zero-Shot API testing...


In [2]:
def classify_comment_with_llm(comment):
    # Prompt Engineering: We provide a specific system instruction to explain the gaming context
    system_prompt = (
        "You are an expert AI moderator for competitive multiplayer games. "
        "Your task is to classify text as either toxic (1) or non-toxic (0). "
        "IMPORTANT: Gaming slang and competitive banter (e.g., 'kill', 'shoot', 'noob', 'destroy') "
        "are normal in gaming contexts and should NOT be flagged as toxic unless accompanied by "
        "real-world threats, severe profanity, or harassment. "
        "Reply ONLY with the number 1 (if toxic) or 0 (if non-toxic). Do not explain."
    )
    
    try:
        # Make the API call to Groq
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Text: \"{comment}\""}
            ],
            model="llama3-8b-8192", # Ultra-fast Llama-3 model hosted on Groq
            temperature=0,          # Set temperature to 0 for deterministic, non-creative outputs
            max_tokens=2            # Restrict output length to just the binary digit
        )
        
        # Extract the model's response and convert it to an integer
        result = chat_completion.choices[0].message.content.strip()
        return int(result) if result in ['0', '1'] else 0
        
    except Exception as e:
        # In case of API timeout or unexpected error, default to Non-Toxic (0)
        return 0

print("Generating predictions via Groq API...")
y_pred = []

# Iterate over the sampled comments with a progress bar
for text in tqdm(X_test_sampled, desc="Calling Llama-3"):
    prediction = classify_comment_with_llm(text)
    y_pred.append(prediction)
    
    # Add a small delay to respect API rate limits
    time.sleep(0.5)

Generating predictions via Groq API...


Calling Llama-3:   0%|          | 0/100 [00:00<?, ?it/s]

In [4]:
print("\n--- ZERO-SHOT LLM (Llama-3) BASELINE RESULTS ---")
# Print the classification report to compare LLM predictions with actual labels
print(classification_report(y_test_sampled, y_pred, target_names=['Non-Toxic', 'Toxic']))


--- ZERO-SHOT LLM (Llama-3) BASELINE RESULTS ---
              precision    recall  f1-score   support

   Non-Toxic       0.88      1.00      0.94        88
       Toxic       0.00      0.00      0.00        12

    accuracy                           0.88       100
   macro avg       0.44      0.50      0.47       100
weighted avg       0.77      0.88      0.82       100



c:\Users\Ayşe Gizem\Desktop\ceng\6. dönem\CENG467-NLP\final\CENG467-Toxicity-Detection-main\CENG467-Toxicity-Detection-main\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ayşe Gizem\Desktop\ceng\6. dönem\CENG467-NLP\final\CENG467-Toxicity-Detection-main\CENG467-Toxicity-Detection-main\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ayşe Gizem\Desktop\ceng\6. dönem\CENG467-NLP\final\CENG467-Toxicity-Detection-main\CENG467-Toxicity-Detection-main\.venv\Lib\site-pac

## 2. Addressing Class Imbalance and Prompt Over-Correction

The initial Zero-Shot experiment yielded a 0.00 F1-score for the Toxic class. This happened for two reasons:
1. **Severe Class Imbalance:** Out of 100 random samples, only 12 were actually toxic.
2. **Prompt Over-Correction:** The system prompt was too lenient, instructing the LLM to tolerate gaming slang so heavily that it ended up classifying *everything* as non-toxic.

To fix this, we will apply **undersampling** to create a perfectly balanced dataset (50% Toxic, 50% Non-Toxic). We will also use a **revised system prompt** that provides stricter, rule-based boundaries between acceptable game jargon and genuine toxicity.

In [5]:
from sklearn.utils import resample

print("Loading data, balancing classes, and revising prompt...")

# 1. Separate majority and minority classes from the original dataframe
df_non_toxic = df[df['toxic'] == 0]
df_toxic = df[df['toxic'] == 1]

# 2. Balance the dataset (Undersampling)
df_non_toxic_downsampled = resample(df_non_toxic, 
                                    replace=False, 
                                    n_samples=len(df_toxic), 
                                    random_state=42)

df_balanced = pd.concat([df_toxic, df_non_toxic_downsampled])

# 3. Sample 100 instances from the BALANCED dataset (50 toxic, 50 non-toxic)
# We limit to 100 to avoid long wait times and API rate limits
df_balanced_sampled = df_balanced.sample(n=100, random_state=42).reset_index(drop=True)

X_test_bal = df_balanced_sampled['comment_text']
y_test_bal = df_balanced_sampled['toxic']

print(f"Sampled Dataset Distribution:\n{y_test_bal.value_counts(normalize=True)}\n")

# 4. Revised Classification Function with a Stricter Prompt
def classify_comment_strict_llm(comment):
    # Prompt Engineering: Clearer boundaries using rule-based logic
    system_prompt = (
        "You are a strict AI moderator for a competitive gaming chat. "
        "Classify the text as toxic (1) or non-toxic (0).\n"
        "Rule 1: Strategic game actions and in-game coordination (e.g., 'kill the enemy', 'shoot the target', 'take their loot') are non-toxic (0).\n"
        "Rule 2: Direct personal attacks, insults (e.g., 'you are trash', 'idiot'), hate speech, severe profanity, and harassment are toxic (1).\n"
        "Reply ONLY with the number 1 or 0. Do not provide any explanations."
    )
    
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Text: \"{comment}\""}
            ],
            model="llama3-8b-8192", 
            temperature=0,          
            max_tokens=2            
        )
        
        result = chat_completion.choices[0].message.content.strip()
        return int(result) if result in ['0', '1'] else 0
        
    except Exception as e:
        return 0

print("Generating predictions via Groq API with Revised Prompt...")
y_pred_bal = []

for text in tqdm(X_test_bal, desc="Calling Llama-3"):
    prediction = classify_comment_strict_llm(text)
    y_pred_bal.append(prediction)
    time.sleep(0.5) # API Rate limit buffer

# 5. Evaluate the Results
print("\n--- BALANCED ZERO-SHOT LLM RESULTS (REVISED PROMPT) ---")
print(classification_report(y_test_bal, y_pred_bal, target_names=['Non-Toxic', 'Toxic']))

Loading data, balancing classes, and revising prompt...
Sampled Dataset Distribution:
toxic
1    0.52
0    0.48
Name: proportion, dtype: float64

Generating predictions via Groq API with Revised Prompt...


Calling Llama-3:   0%|          | 0/100 [00:00<?, ?it/s]


--- BALANCED ZERO-SHOT LLM RESULTS (REVISED PROMPT) ---
              precision    recall  f1-score   support

   Non-Toxic       0.48      1.00      0.65        48
       Toxic       0.00      0.00      0.00        52

    accuracy                           0.48       100
   macro avg       0.24      0.50      0.32       100
weighted avg       0.23      0.48      0.31       100



c:\Users\Ayşe Gizem\Desktop\ceng\6. dönem\CENG467-NLP\final\CENG467-Toxicity-Detection-main\CENG467-Toxicity-Detection-main\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ayşe Gizem\Desktop\ceng\6. dönem\CENG467-NLP\final\CENG467-Toxicity-Detection-main\CENG467-Toxicity-Detection-main\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ayşe Gizem\Desktop\ceng\6. dönem\CENG467-NLP\final\CENG467-Toxicity-Detection-main\CENG467-Toxicity-Detection-main\.venv\Lib\site-pac